In [1]:
%pip install prophet wandb -Uq

import os
import getpass

import wandb

_wandb_api_key_env = os.environ.get("WANDB_API_KEY")
os.environ.pop("WANDB_API_KEY", None)
try:
    wandb.logout()
except Exception:
    pass

WANDB_ENTITY = "toberi23-free-university-of-tbilisi-"
WANDB_PROJECT = "Walmart-Recruiting-Store-Sales-Forecasting"

wandb_key = _wandb_api_key_env or getpass.getpass("Paste your W&B API key: ").strip()
wandb.login(key=wandb_key, relogin=True, force=True, verify=True)

who = wandb.Api().viewer
print("Logged in as:", who.username, "| entity:", who.entity)


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/macbookpro/.netrc
wandb: Currently logged in as: lmars23 (toberi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Logged in as: lmars23 | entity: toberi23-free-university-of-tbilisi-


In [2]:
import os, time, pickle, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin

from prophet import Prophet
import logging
logging.getLogger("cmdstanpy").setLevel(logging.WARNING)
logging.getLogger("prophet").setLevel(logging.WARNING)

import wandb

SEED = 42
np.random.seed(SEED)

WANDB_GROUP = "Prophet_Training"
pd.set_option("display.width", 200)

Importing plotly failed. Interactive plots will not work.


In [3]:
_LOCAL_DATA_DIR = os.path.join("data", "walmart-recruiting-store-sales-forecasting")
_KAGGLE_DATA_DIR = "/kaggle/input/competitions/walmart-recruiting-store-sales-forecasting"

COMP = (
    os.environ.get("WALMART_DATA_DIR")
    or (_LOCAL_DATA_DIR if os.path.isdir(_LOCAL_DATA_DIR) else _KAGGLE_DATA_DIR)
)
print("Reading competition data from:", COMP)

def load_merged(kind: str = "train") -> pd.DataFrame:
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    base = pd.read_csv(f"{COMP}/{kind}.csv.zip")
    base["Date"] = pd.to_datetime(base["Date"])
    stores = pd.read_csv(f"{COMP}/stores.csv")
    feats = pd.read_csv(f"{COMP}/features.csv.zip")
    feats["Date"] = pd.to_datetime(feats["Date"])
    feats = feats.drop(columns=["IsHoliday"])
    return (
        base.merge(stores, on="Store", how="left")
            .merge(feats, on=["Store", "Date"], how="left")
            .sort_values(["Store", "Dept", "Date"])
            .reset_index(drop=True)
    )

df_train = load_merged("train")
df_test = load_merged("test")
print("Train:", df_train.shape, "| Test:", df_test.shape)
df_train.head()

Reading competition data from: data/walmart-recruiting-store-sales-forecasting
Train: (421570, 16) | Test: (115064, 15)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,24924.50,False,A,151315,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106
1,1,1,2010-02-12,46039.49,True,A,151315,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106
2,1,1,2010-02-19,41595.55,False,A,151315,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106
3,1,1,2010-02-26,19403.54,False,A,151315,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106
4,1,1,2010-03-05,21827.90,False,A,151315,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106


In [4]:
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday, dtype=bool), 5.0, 1.0)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))

assert abs(wmae([10, 20], [8, 15], [True, False]) - 15 / 6) < 1e-9

VAL_START = pd.Timestamp("2011-11-04")
VAL_END = pd.Timestamp("2012-07-27")

cv_train = df_train[df_train.Date < VAL_START].copy()
cv_val = df_train[(df_train.Date >= VAL_START) & (df_train.Date <= VAL_END)].copy()

print("CV train:", cv_train.shape, cv_train.Date.min().date(), "->", cv_train.Date.max().date())
print("CV val  :", cv_val.shape, cv_val.Date.min().date(), "->", cv_val.Date.max().date())
print("Val holiday weeks:", cv_val.loc[cv_val.IsHoliday, "Date"].dt.strftime("%Y-%m-%d").unique().tolist())

CV train: (267184, 16) 2010-02-05 -> 2011-10-28
CV val  : (115856, 16) 2011-11-04 -> 2012-07-27
Val holiday weeks: ['2011-11-25', '2011-12-30', '2012-02-10']


In [5]:
MARKDOWN_COLS = [f"MarkDown{i}" for i in range(1, 6)]
MEDIAN_FILL_COLS = ["CPI", "Unemployment", "Temperature", "Fuel_Price"]

class DataCleaner(BaseEstimator, TransformerMixin):

    def __init__(self, markdown_cols=None, median_cols=None):
        self.markdown_cols = markdown_cols if markdown_cols is not None else MARKDOWN_COLS
        self.median_cols = median_cols if median_cols is not None else MEDIAN_FILL_COLS

    def fit(self, X, y=None):
        self.medians_ = {c: X[c].median() for c in self.median_cols}
        return self

    def transform(self, X):
        X = X.copy()
        X[self.markdown_cols] = X[self.markdown_cols].fillna(0.0)
        for c in self.median_cols:
            X[c] = X[c].fillna(self.medians_[c])
        return X

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="Prophet_Data_Cleaning", job_type="data-cleaning")

cleaner = DataCleaner()
cleaner.fit(cv_train)

cv_train_clean = cleaner.transform(cv_train)
cv_val_clean = cleaner.transform(cv_val)

missing_report = cv_train_clean[cleaner.markdown_cols + cleaner.median_cols].isna().mean()
print("Learned fill medians:", cleaner.medians_)
print("\nRemaining missing % after cleaning:\n", missing_report)

wandb.config.update({"markdown_fill": 0, **{f"median_{c}": v for c, v in cleaner.medians_.items()}})
wandb.log({"post_clean_missing_pct_total": float(missing_report.sum())})
run.finish()

Learned fill medians: {'CPI': np.float64(182.0774691), 'Unemployment': np.float64(8.099), 'Temperature': np.float64(62.97), 'Fuel_Price': np.float64(3.046)}

Remaining missing % after cleaning:
 MarkDown1       0.0
MarkDown2       0.0
MarkDown3       0.0
MarkDown4       0.0
MarkDown5       0.0
CPI             0.0
Unemployment    0.0
Temperature     0.0
Fuel_Price      0.0
dtype: float64


post_clean_missing_pct_total,▁
post_clean_missing_pct_total,0


In [6]:
def _build_holidays_table(lower_window=-3, upper_window=1):
    raw = {
        "Super Bowl":   ["2010-02-12", "2011-02-11", "2012-02-10", "2013-02-08"],
        "Labor Day":    ["2010-09-10", "2011-09-09", "2012-09-07", "2013-09-06"],
        "Thanksgiving": ["2010-11-26", "2011-11-25", "2012-11-23", "2013-11-29"],
        "Christmas":    ["2010-12-31", "2011-12-30", "2012-12-28", "2013-12-27"],
    }
    rows = []
    for name, dates in raw.items():
        for d in dates:
            rows.append({"holiday": name, "ds": pd.Timestamp(d),
                         "lower_window": lower_window, "upper_window": upper_window})
    return pd.DataFrame(rows)

class HolidayFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, lower_window=-3, upper_window=1):
        self.lower_window = lower_window
        self.upper_window = upper_window

    def fit(self, X=None, y=None):
        self.holidays_df_ = _build_holidays_table(self.lower_window, self.upper_window)
        return self

    def transform(self, X):
        return X

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="Prophet_Feature_Engineering", job_type="feature-engineering",
                  config={"holiday_window": [-3, 1]})

engineer = HolidayFeatureEngineer(lower_window=-3, upper_window=1)
engineer.fit()
print(engineer.holidays_df_)

wandb.log({"n_holiday_rows": len(engineer.holidays_df_)})
run.finish()

         holiday         ds  lower_window  upper_window
0     Super Bowl 2010-02-12            -3             1
1     Super Bowl 2011-02-11            -3             1
2     Super Bowl 2012-02-10            -3             1
3     Super Bowl 2013-02-08            -3             1
4      Labor Day 2010-09-10            -3             1
5      Labor Day 2011-09-09            -3             1
6      Labor Day 2012-09-07            -3             1
7      Labor Day 2013-09-06            -3             1
8   Thanksgiving 2010-11-26            -3             1
9   Thanksgiving 2011-11-25            -3             1
10  Thanksgiving 2012-11-23            -3             1
11  Thanksgiving 2013-11-29            -3             1
12     Christmas 2010-12-31            -3             1
13     Christmas 2011-12-30            -3             1
14     Christmas 2012-12-28            -3             1
15     Christmas 2013-12-27            -3             1


n_holiday_rows,▁
n_holiday_rows,16


In [7]:
CANDIDATE_REGRESSOR_COLS = ["IsHoliday"] + MARKDOWN_COLS + ["Temperature", "Fuel_Price", "CPI", "Unemployment"]
STRUCTURALLY_EXCLUDED_COLS = list(MARKDOWN_COLS)

class RegressorSelector(BaseEstimator, TransformerMixin):

    def __init__(self, candidate_cols=None, structurally_excluded=None):
        self.candidate_cols = candidate_cols if candidate_cols is not None else CANDIDATE_REGRESSOR_COLS
        self.structurally_excluded = structurally_excluded if structurally_excluded is not None else STRUCTURALLY_EXCLUDED_COLS

    def fit(self, X, y=None):
        target = y if y is not None else X["Weekly_Sales"]
        corr_frame = X[self.candidate_cols].copy()
        if "IsHoliday" in corr_frame.columns:
            corr_frame["IsHoliday"] = corr_frame["IsHoliday"].astype(int)
        self.correlations_ = corr_frame.corrwith(target).abs().sort_values(ascending=False)
        self.selected_regressors_ = [c for c in self.candidate_cols if c not in self.structurally_excluded]
        return self

    def transform(self, X):
        key_cols = [c for c in ["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"] if c in X.columns]
        keep = key_cols + [c for c in self.selected_regressors_ if c not in key_cols]
        return X[keep]

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="Prophet_Feature_Selection", job_type="feature-selection")

selector = RegressorSelector()
selector.fit(cv_train_clean)

print("Train-only |correlation| with Weekly_Sales:\n", selector.correlations_)
print("\nSelected regressors:", selector.selected_regressors_)
print("Dropped regressors (structurally excluded):", selector.structurally_excluded)

wandb.log({
    "n_candidate_regressors": len(selector.candidate_cols),
    "n_selected_regressors": len(selector.selected_regressors_),
    **{f"corr_{c}": float(selector.correlations_.get(c, np.nan)) for c in selector.candidate_cols},
})
run.finish()

Train-only |correlation| with Weekly_Sales:
 CPI             0.022598
Unemployment    0.021444
IsHoliday       0.009405
Temperature     0.000785
Fuel_Price      0.000231
MarkDown1            NaN
MarkDown2            NaN
MarkDown3            NaN
MarkDown4            NaN
MarkDown5            NaN
dtype: float64

Selected regressors: ['IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
Dropped regressors (structurally excluded): ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']


corr_CPI,▁
corr_Fuel_Price,▁
corr_IsHoliday,▁
corr_Temperature,▁
corr_Unemployment,▁
n_candidate_regressors,▁
n_selected_regressors,▁
+5,...
corr_CPI,0.0226
corr_Fuel_Price,0.00023
corr_IsHoliday,0.00941


In [8]:
SAMPLE_N = 150

series_mean = cv_train.groupby(["Store", "Dept"])["Weekly_Sales"].mean()
series_keys_all = cv_train[["Store", "Dept"]].drop_duplicates()
series_keys_all["quartile"] = pd.qcut(
    series_keys_all.apply(lambda r: series_mean[(r.Store, r.Dept)], axis=1), 4, labels=False, duplicates="drop"
)

sample_keys = (
    series_keys_all.groupby("quartile", group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), SAMPLE_N // series_keys_all.quartile.nunique()), random_state=SEED))
    [["Store", "Dept"]]
    .reset_index(drop=True)
)
print("Sampled series:", len(sample_keys), "/", len(series_keys_all))
sample_keys.head()

Sampled series: 148 / 3254


,Store,Dept
0,17,59
1,37,24
2,16,49
3,20,77
4,35,78


In [9]:
MIN_HISTORY_WEEKS = 52

class SeasonalNaiveFallback:

    def __init__(self, min_history_weeks=MIN_HISTORY_WEEKS):
        self.min_history_weeks = min_history_weeks

    def is_needed(self, history_df):
        if len(history_df) == 0:
            return True
        n_hist = history_df["Date"].nunique()
        series_mean = history_df["Weekly_Sales"].mean()
        return n_hist < self.min_history_weeks or history_df["Weekly_Sales"].std() == 0 or np.isnan(series_mean)

    def predict(self, history_df, future_df, global_fallback_mean):
        series_mean = history_df["Weekly_Sales"].mean() if len(history_df) else global_fallback_mean
        fallback_value = series_mean if not np.isnan(series_mean) else global_fallback_mean

        lag = history_df[["Date", "Weekly_Sales"]].rename(columns={"Date": "lag_date", "Weekly_Sales": "snaive"})
        fd = future_df.copy()
        fd["lag_date"] = fd["Date"] - pd.Timedelta(weeks=52)
        fd = fd.merge(lag, on="lag_date", how="left")
        return fd["snaive"].fillna(fallback_value).values

class ProphetSeriesModel(BaseEstimator, RegressorMixin):

    def __init__(self, changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                 holidays_prior_scale=10.0, seasonality_mode="additive",
                 changepoint_range=0.8, n_changepoints=25, yearly_seasonality_order=10,
                 use_regressors=True, regressor_cols=None, holidays_df=None,
                 min_history_weeks=MIN_HISTORY_WEEKS):
        self.changepoint_prior_scale = changepoint_prior_scale
        self.seasonality_prior_scale = seasonality_prior_scale
        self.holidays_prior_scale = holidays_prior_scale
        self.seasonality_mode = seasonality_mode
        self.changepoint_range = changepoint_range
        self.n_changepoints = n_changepoints
        self.yearly_seasonality_order = yearly_seasonality_order
        self.use_regressors = use_regressors
        self.regressor_cols = regressor_cols or []
        self.holidays_df = holidays_df
        self.min_history_weeks = min_history_weeks

    def fit(self, X, y=None):
        self.fallback_ = SeasonalNaiveFallback(self.min_history_weeks)
        self.history_ = X
        self.needs_fallback_ = self.fallback_.is_needed(X)

        if not self.needs_fallback_:
            prophet_df = X.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["ds", "y"] + self.regressor_cols]
            self.model_ = Prophet(
                holidays=self.holidays_df,
                changepoint_prior_scale=self.changepoint_prior_scale,
                seasonality_prior_scale=self.seasonality_prior_scale,
                holidays_prior_scale=self.holidays_prior_scale,
                seasonality_mode=self.seasonality_mode,
                changepoint_range=self.changepoint_range,
                n_changepoints=self.n_changepoints,
                yearly_seasonality=self.yearly_seasonality_order,
                weekly_seasonality=False,
                daily_seasonality=False,
            )
            if self.use_regressors:
                for col in self.regressor_cols:
                    self.model_.add_regressor(col)
            self.model_.fit(prophet_df)
        return self

    def predict(self, X, global_fallback_mean=0.0):
        if self.needs_fallback_:
            return self.fallback_.predict(self.history_, X, global_fallback_mean)

        cols = ["ds"] + (self.regressor_cols if self.use_regressors else [])
        future = X.rename(columns={"Date": "ds"})[cols]
        forecast = self.model_.predict(future)
        return forecast["yhat"].values

CHRISTMAS_WEEK52_ANCHORS = {
    2010: pd.Timestamp("2010-12-31"),
    2011: pd.Timestamp("2011-12-30"),
    2012: pd.Timestamp("2012-12-28"),
}
TARGET_YEAR = 2012

class ChristmasWeekShiftAdjuster:

    def __init__(self, bulge_threshold=1.10, christmas_anchors=None, target_year=TARGET_YEAR):
        self.bulge_threshold = bulge_threshold
        self.christmas_anchors = christmas_anchors if christmas_anchors is not None else CHRISTMAS_WEEK52_ANCHORS
        self.target_year = target_year

    def fit(self, train_df):
        self.train_start_ = train_df.groupby(["Store", "Dept"])["Date"].min()
        self.dec_cutoffs_ = {year: anchor - pd.Timedelta(weeks=4) for year, anchor in self.christmas_anchors.items()}
        return self

    def adjust(self, X, preds):
        pred_df = X[["Store", "Dept", "Date"]].copy()
        pred_df["Weekly_Sales"] = preds
        pred_df["_row"] = np.arange(len(pred_df))

        adjusted = preds.copy()
        self.last_stats_ = {"n_bulge_departments": 0, "n_shifted_two_year": 0,
                             "n_shifted_one_year": 0, "total_abs_sales_shifted": 0.0}

        for year, anchor in self.christmas_anchors.items():
            if year != self.target_year:
                continue
            week_dates = {w: anchor - pd.Timedelta(weeks=52 - w) for w in [48, 49, 50, 51, 52]}
            week_lookup = {v: k for k, v in week_dates.items()}
            in_window = pred_df["Date"].isin(week_lookup)
            if not in_window.any():
                continue

            sub = pred_df.loc[in_window].copy()
            sub["week"] = sub["Date"].map(week_lookup)
            wide = sub.pivot_table(index=["Store", "Dept"], columns="week", values="Weekly_Sales")
            wide = wide.dropna(subset=[48, 49, 50, 51, 52])
            if wide.empty:
                continue

            avg_49_51 = wide[[49, 50, 51]].mean(axis=1)
            avg_48_52 = wide[[48, 52]].mean(axis=1)
            has_bulge = avg_49_51 >= self.bulge_threshold * avg_48_52

            train_start = self.train_start_.reindex(wide.index)
            has_two_years = train_start <= self.dec_cutoffs_.get(year - 1, pd.Timestamp.min)
            has_one_year = (train_start <= self.dec_cutoffs_.get(year, pd.Timestamp.min)) & (~has_two_years)

            shift_fraction = pd.Series(0.0, index=wide.index)
            shift_fraction[has_two_years] = 2.5 / 7
            shift_fraction[has_one_year] = 2.0 / 7
            shift_fraction[~has_bulge] = 0.0

            to_shift = shift_fraction[shift_fraction > 0].index
            if len(to_shift) == 0:
                continue

            shifted = wide.copy()
            for key in to_shift:
                vals = wide.loc[key, [48, 49, 50, 51, 52]].to_numpy(dtype=float)
                s = shift_fraction[key]
                shifted.loc[key] = [(1 - s) * vals[i] + s * vals[(i - 1) % 5] for i in range(5)]

            shifted_long = shifted.loc[to_shift].stack().rename("Weekly_Sales_shifted").reset_index()
            shifted_long["Date"] = shifted_long["week"].map(week_dates)
            merged = sub.merge(shifted_long[["Store", "Dept", "Date", "Weekly_Sales_shifted"]],
                                on=["Store", "Dept", "Date"], how="left")
            merged = merged.dropna(subset=["Weekly_Sales_shifted"])
            adjusted[merged["_row"].to_numpy()] = merged["Weekly_Sales_shifted"].to_numpy()

            delta_abs = (shifted.loc[to_shift] - wide.loc[to_shift]).abs().to_numpy().sum()
            self.last_stats_["n_bulge_departments"] += int(has_bulge.sum())
            self.last_stats_["n_shifted_two_year"] += int((shift_fraction == 2.5 / 7).sum())
            self.last_stats_["n_shifted_one_year"] += int((shift_fraction == 2.0 / 7).sum())
            self.last_stats_["total_abs_sales_shifted"] += float(delta_abs)

        return adjusted

class ProphetForecastPipeline(BaseEstimator, RegressorMixin):

    def __init__(self, changepoint_prior_scale=0.05, seasonality_prior_scale=10.0,
                 holidays_prior_scale=10.0, seasonality_mode="additive",
                 changepoint_range=0.8, n_changepoints=25, yearly_seasonality_order=10,
                 use_regressors=True, min_history_weeks=MIN_HISTORY_WEEKS,
                 holiday_window=(-3, 1), use_christmas_shift=True, christmas_bulge_threshold=1.10):
        self.changepoint_prior_scale = changepoint_prior_scale
        self.seasonality_prior_scale = seasonality_prior_scale
        self.holidays_prior_scale = holidays_prior_scale
        self.seasonality_mode = seasonality_mode
        self.changepoint_range = changepoint_range
        self.n_changepoints = n_changepoints
        self.yearly_seasonality_order = yearly_seasonality_order
        self.use_regressors = use_regressors
        self.min_history_weeks = min_history_weeks
        self.holiday_window = holiday_window
        self.use_christmas_shift = use_christmas_shift
        self.christmas_bulge_threshold = christmas_bulge_threshold

    def fit(self, X, y=None):
        self.cleaner_ = DataCleaner().fit(X)
        X_clean = self.cleaner_.transform(X)

        self.engineer_ = HolidayFeatureEngineer(*self.holiday_window).fit()

        self.selector_ = RegressorSelector().fit(X_clean)
        self.train_ = self.selector_.transform(X_clean)

        self.global_fallback_mean_ = self.train_["Weekly_Sales"].mean()

        self.series_models_ = {}
        for (store, dept), history in self.train_.groupby(["Store", "Dept"]):
            series_model = self._make_series_model()
            series_model.fit(history.sort_values("Date"))
            self.series_models_[(store, dept)] = series_model

        if self.use_christmas_shift:
            self.christmas_adjuster_ = ChristmasWeekShiftAdjuster(
                bulge_threshold=self.christmas_bulge_threshold
            ).fit(self.train_)
        return self

    def _make_series_model(self):
        return ProphetSeriesModel(
            changepoint_prior_scale=self.changepoint_prior_scale,
            seasonality_prior_scale=self.seasonality_prior_scale,
            holidays_prior_scale=self.holidays_prior_scale,
            seasonality_mode=self.seasonality_mode,
            changepoint_range=self.changepoint_range,
            n_changepoints=self.n_changepoints,
            yearly_seasonality_order=self.yearly_seasonality_order,
            use_regressors=self.use_regressors,
            regressor_cols=self.selector_.selected_regressors_,
            holidays_df=self.engineer_.holidays_df_,
            min_history_weeks=self.min_history_weeks,
        )

    def predict(self, X):
        X_clean = self.cleaner_.transform(X)
        X_sel = self.selector_.transform(X_clean)

        unseen_fallback_ = SeasonalNaiveFallback(self.min_history_weeks)

        preds = np.zeros(len(X_sel))
        for (store, dept), future in X_sel.groupby(["Store", "Dept"]):
            future_sorted = future.sort_values("Date")
            series_model = self.series_models_.get((store, dept))
            if series_model is not None:
                preds[future_sorted.index.values] = series_model.predict(future_sorted, self.global_fallback_mean_)
            else:
                history = self.train_[(self.train_.Store == store) & (self.train_.Dept == dept)].sort_values("Date")
                preds[future_sorted.index.values] = unseen_fallback_.predict(
                    history, future_sorted, self.global_fallback_mean_
                )

        if self.use_christmas_shift:
            preds = self.christmas_adjuster_.adjust(X_sel, preds)

        return preds

def evaluate_config(config, train_full, val_full, keys, verbose=False):
    val_subset = val_full.merge(keys, on=["Store", "Dept"])

    pipeline = ProphetForecastPipeline(**config)
    pipeline.fit(train_full)
    preds = pipeline.predict(val_subset)

    score = wmae(val_subset["Weekly_Sales"], preds, val_subset["IsHoliday"])
    if verbose:
        print(f"WMAE on {len(keys)} series: {score:.2f}")
    return score

def run_stage(stage_name, configs, base_config, keys=None):
    keys = sample_keys if keys is None else keys
    results = []
    for cfg_overrides in configs:
        cfg = {**base_config, **cfg_overrides}
        tag = "_".join(f"{k}{v}" for k, v in cfg_overrides.items())
        run_name = f"Prophet_Training_Sweep_{stage_name}_{tag}"

        run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name=run_name, job_type="training-sweep",
                          config={**cfg, "stage": stage_name, "sample_size": len(keys)}, reinit=True)
        t0 = time.time()
        score = evaluate_config(cfg, cv_train, cv_val, keys)
        elapsed = time.time() - t0
        wandb.log({"wmae": score, "seconds": elapsed})
        run.finish()

        results.append({**cfg_overrides, "wmae": score, "seconds": elapsed})
        print(f"{run_name}: WMAE={score:.2f} ({elapsed:.1f}s)")

    return pd.DataFrame(results).sort_values("wmae").reset_index(drop=True)

In [ ]:
best_config = {
    "changepoint_prior_scale": 0.05,
    "seasonality_prior_scale": 0.1,
    "holidays_prior_scale": 1.0,
    "seasonality_mode": "additive",
    "changepoint_range": 0.95,
    "n_changepoints": 15,
    "yearly_seasonality_order": 20,
    "use_regressors": False,
    "use_christmas_shift": True,
}
full_wmae = 2138.00
print("Using hardcoded final config (README):", best_config)

In [ ]:
final_pipeline = ProphetForecastPipeline(**best_config)
final_pipeline.fit(df_train)

with open("prophet_pipeline.pkl", "wb") as f:
    pickle.dump(final_pipeline, f)

run = wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, group=WANDB_GROUP, name="Prophet_Pipeline_Save", job_type="save-pipeline",
                  config={**best_config, "cv_wmae_full": full_wmae})

artifact = wandb.Artifact("prophet_forecast_pipeline", type="model",
                           metadata={**best_config, "cv_wmae_full": full_wmae})
artifact.add_file("prophet_pipeline.pkl")
run.log_artifact(artifact)
wandb.log({"cv_wmae_full": full_wmae})
run.finish()

print("Saved prophet_pipeline.pkl and logged it as a W&B Artifact.")